# Decoder (GPT-2 para generación de texto)

In [ ]:
# Bloque 1: Instalación de dependencias
# Instalamos Hugging Face transformers y datasets
!pip install transformers datasets


In [ ]:
# Bloque 2: Importar librerías necesarias
# Cargamos GPT-2 para generación de texto
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [ ]:
# Bloque 3: Cargar tokenizer y modelo decoder (GPT-2 pequeño)
# GPT-2 es un modelo "decoder-only", especializado en predecir la siguiente palabra.
# Usamos la versión "distilgpt2" que es más liviana y rápida para entrenar/demo.
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Bloque 4: Crear un pipeline de generación de texto
# Este pipeline nos permite darle un texto inicial (prompt)
# y el modelo genera automáticamente la continuación.
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)


Device set to use cpu


In [ ]:
# Bloque 5: Ejemplo de generación de texto
# Aquí el modelo predice cómo continuar la frase dada.
output = generator("Once upon a time, in a faraway land,",
                   max_length=50,   # longitud máxima del texto generado
                   num_return_sequences=2,  # cuántas alternativas generar
                   temperature=0.7) # controla la creatividad (bajo = más preciso, alto = más creativo)

print(output)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Once upon a time, in a faraway land, the entire family would have lost their children.\n\n\n\n"I remember that I was in the same family in my village, and I remember that my parents used to call me my baby," said the father of a child.\nBut when asked about it, the father of the girl, who died on the way home, said, "I never heard of my mother. I never thought of anything else."\nThe father of the girl, who died a few weeks ago, said, "I never heard of my mother."\nHe recalled that when he was a girl, he went to work and spent a few weeks working in a farm.\n"I was in a very small village," said the father of the girl, who died in the hospital this morning. "My mother was very young. She was very young. She was very young; she was very young. I\'ve never seen her in the village."\nIn the village, the father of the girl, who died in the hospital, said, "I was the only one to die."\n"My mother was very young, and she was very young. She was very young. She was very y

In [ ]:
# Bloque 6: Usar tu propio dataset (opcional)
# pueden cargar un CSV con una columna de texto y "afinar" el modelo GPT-2.
# Ejemplo: dataset de frases para entrenar estilo específico.

from datasets import Dataset
import pandas as pd

# Ejemplo de un dataset propio )
data = {"text": [
    "El clima hoy está muy soleado picante.",
    "La inteligencia artificial está transformando el mundo.",
    "Había una vez un dragón que vivía en las montañas."
]}
df = pd.DataFrame(data)

dataset = Dataset.from_pandas(df)
print(dataset[0])


{'text': 'El clima hoy está muy soleado picante.'}


In [ ]:
def tokenize_function(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=50
    )
    # 👇 usamos input_ids también como labels
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)



Map:   0%|          | 0/3 [00:00<?, ? examples/s]

ValueError: Asking to pad but the tokenizer does not have a padding token. Please select a token to use as `pad_token` `(tokenizer.pad_token = tokenizer.eos_token e.g.)` or add a new pad token via `tokenizer.add_special_tokens({'pad_token': '[PAD]'})`.

In [ ]:
# Bloque 8: Pequeño fine-tuning (opcional)
# Si quieres, puedes entrenar GPT-2 con tu dataset para adaptarlo.
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_decoder",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=5,
    report_to="none"  # evita usar wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()


NameError: name 'tokenized_dataset' is not defined

In [ ]:
input_text = "The future of German Castro Caicedo is"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_length=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


The future of German Castro Caicedo is a matter of time before the Cuban government can take over the country.



















# Ejemplo 1: Decoder para completar texto (GPT-2)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Modelo pequeño para que no tarde mucho
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Aseguramos pad_token
tokenizer.pad_token = tokenizer.eos_token

# Entrada del usuario
input_text = "In the future, AMAZONAS biodiversity will"
inputs = tokenizer(input_text, return_tensors="pt")

# Generación
outputs = model.generate(**inputs, max_length=40, num_return_sequences=3, do_sample=True, top_k=50)
for i, output in enumerate(outputs):
    print(f"\n✨ Resultado {i+1}: {tokenizer.decode(output, skip_special_tokens=True)}")


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



✨ Resultado 1: In the future, AMAZONAS biodiversity will begin to be assessed, compared to some parts of the world and in some parts of the world.








L

✨ Resultado 2: In the future, AMAZONAS biodiversity will remain under threat.





Follow @thecoffordnews

Read or Share this story: http://lohud

✨ Resultado 3: In the future, AMAZONAS biodiversity will be a real benefit. What’s up in your mind:

I’ve just got some ideas…
Here's just one


# Ejemplo 2: Decoder para responder preguntas simples (Q&A estilo generativo)

In [ ]:
question = "What is the best superhero?"

inputs = tokenizer("Q: " + question + "\nA:", return_tensors="pt")
outputs = model.generate(**inputs, max_length=30, num_return_sequences=1)

print("❓ Pregunta:", question)
print("💡 Respuesta generada:", tokenizer.decode(outputs[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


❓ Pregunta: What is the best superhero?
💡 Respuesta generada: Q: What is the best superhero?
A: I think it's a very good superhero. I think it's a very good superhero. I


# Probando otro modelo mas robusto

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

inputs = tokenizer("What is the meaning of life?", return_tensors="pt")
outputs = model.generate(**inputs, max_length=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))


life is a state of being


# Ejemplo 3: Decoder para escribir estilo creativo

In [ ]:
prompt = "Drown in your dreams at the end of the cup,"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=50,
    num_return_sequences=2,   # Queremos dos textos diferentes
    do_sample=True,           # Sampling activado
    temperature=0.3,          # Controla creatividad (más alto = más creativo)
    top_p=0.95                # Nucleus sampling
)

for i, output in enumerate(outputs):
    print(f"\n📖 Historia {i+1}: {tokenizer.decode(output, skip_special_tokens=True)}")



📖 Historia 1: slams your eyes with a sigh of relief, and screams at your dreams.

📖 Historia 2: if you're a fan of the game, you'll love it.


# Ejemplo 4: Decoder para chat interactivo en Colab

In [ ]:
def chat_with_model():
    print("💬 Welcome at the GPT2-mini. Write 'exit' to finish it.\n")
    while True:
        user_input = input("Tú: ")
        if user_input.lower() == "exit":
            break
        inputs = tokenizer(user_input, return_tensors="pt")
        outputs = model.generate(**inputs, max_length=60, pad_token_id=tokenizer.eos_token_id)
        print("🤖 Modelo:", tokenizer.decode(outputs[0], skip_special_tokens=True))

chat_with_model()


💬 Welcome at the GPT2-mini. Write 'exit' to finish it.

Tú: Hey baby
🤖 Modelo: Hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby, hey baby,
Tú: calm down fire
🤖 Modelo: calm down fire
Tú: dream and scresm
🤖 Modelo: dream and scresm
Tú: exit


# Ejemplo 5: Decoder con dataset propio

In [ ]:
from datasets import Dataset

# Pequeño dataset de frases propias
texts = [
    "Artificial Intelligence will lost the world.",
    "Machine learning is a subset of AI.",
    "Transformers are powerful architectures."
]

dataset = Dataset.from_dict({"text": texts})

# Tokenización con labels
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=30)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(tokenized_dataset[0])


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

{'text': 'Artificial Intelligence will change the world.', 'input_ids': [24714, 5869, 2825, 1433, 56, 483, 8, 296, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [24714, 5869, 2825, 1433, 56, 483, 8, 296, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

# Cargar un modelo pequeño para pruebas (rápido en Colab)
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

training_args = TrainingArguments(
    output_dir="./results_decoder",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    logging_steps=5,
    report_to="none"  # evita usar wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()





/tmp/ipython-input-4212476431.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,4.894900


TrainOutput(global_step=6, training_loss=4.6264564990997314, metrics={'train_runtime': 24.9093, 'train_samples_per_second': 0.361, 'train_steps_per_second': 0.241, 'total_flos': 68896604160.0, 'train_loss': 4.6264564990997314, 'epoch': 3.0})

In [ ]:
prompt = "Artificial Intelligence"
inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=40,
    num_return_sequences=1,
    temperature=0.3,
    top_p=0.9
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Artificial Intelligence și și și și și și și și și și și și și și și și și și și și și și și și și și și și și și și și și și și


Próximos pasos:
Probar modelos como:

distilgpt2 → demo rápido.

gpt2 o facebook/opt-125m → para resultados un poco mejores.

bigscience/bloom-560m → para ver generación en español.